In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# เขียนโปรแกรม Qiskit Serverless แรกของคุณ

<Accordion>
<AccordionItem title="Package versions">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</AccordionItem>
</Accordion>
> **Tip:** **Qiskit Serverless กำลังได้รับการอัปเกรด และฟีเจอร์ต่าง ๆ กำลังเปลี่ยนแปลงอย่างรวดเร็ว** ในช่วงการพัฒนานี้ สามารถดู release notes และเอกสารล่าสุดได้ที่หน้า [Qiskit Serverless GitHub](https://qiskit.github.io/qiskit-serverless/index.html)

ตัวอย่างนี้สาธิตวิธีใช้เครื่องมือ `qiskit-serverless` เพื่อสร้างโปรแกรม transpilation แบบ parallel จากนั้นใช้ `qiskit-ibm-catalog` เพื่ออัปโหลดโปรแกรมไปยัง IBM Quantum Platform ให้ใช้เป็น remote service ที่นำกลับมาใช้ซ้ำได้

## ภาพรวมของ workflow
1. สร้างไดเรกทอรีในเครื่องและไฟล์โปรแกรมเปล่า (`./source_files/transpile_remote.py`)
1. เพิ่มโค้ดเข้าไปในโปรแกรมที่เมื่ออัปโหลดไปยัง Qiskit Serverless จะทำการ transpile circuit
1. ใช้ `qiskit-ibm-catalog` เพื่อ authenticate กับ Qiskit Serverless
1. อัปโหลดโปรแกรมไปยัง Qiskit Serverless

หลังจากอัปโหลดโปรแกรมแล้ว สามารถรันโปรแกรมเพื่อ transpile circuit ได้โดยทำตามคู่มือ [รัน Qiskit Serverless workload แรกจากระยะไกล](/guides/serverless-run-first-workload)

## ตัวอย่าง: Remote transpilation ด้วย Qiskit Serverless
ตัวอย่างนี้จะพาคุณสร้างและเพิ่มโค้ดลงในไฟล์โปรแกรมที่เมื่ออัปโหลดไปยัง Qiskit Serverless จะทำการ transpile `circuit` กับ `backend` และ `optimization_level` ที่กำหนด

:::tip

Qiskit Serverless ต้องการให้ตั้งค่าไฟล์ `.py` ของ workload ลงในไดเรกทอรีที่กำหนด โครงสร้างต่อไปนี้เป็นตัวอย่างของ best practice:

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

### เพิ่มโค้ดเพื่อรับ arguments ของโปรแกรม
ตอนนี้เพิ่มโค้ดต่อไปนี้ลงในไฟล์โปรแกรม ซึ่งจะตั้งค่า arguments ของโปรแกรม

`transpile_remote.py` เริ่มต้นมี input สามตัว ได้แก่ `circuits`, `backend_name`, และ `optimization_level` ปัจจุบัน Serverless รับได้เฉพาะ input และ output ที่ serialize ได้เท่านั้น ด้วยเหตุนี้จึงไม่สามารถส่ง `backend` โดยตรงได้ ให้ใช้ `backend_name` เป็น string แทน

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

### เพิ่มโค้ดที่เรียก backend
เพิ่มโค้ดต่อไปนี้ลงในไฟล์โปรแกรม ซึ่งจะเรียก backend ด้วย `QiskitRuntimeService`

โค้ดต่อไปนี้ถือว่าคุณได้ทำตามกระบวนการบันทึก credentials โดยใช้ `QiskitRuntimeService.save_account` แล้ว และจะโหลด account เริ่มต้นที่บันทึกไว้เว้นแต่จะระบุไว้เป็นอย่างอื่น ดูข้อมูลเพิ่มเติมได้ที่ [บันทึกข้อมูล login ของคุณ](/guides/save-credentials) และ [เริ่มต้นใช้งาน Qiskit Runtime service account](/guides/initialize-account)

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this
# locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

### เพิ่มโค้ดเพื่อ transpile
สุดท้าย เพิ่มโค้ดต่อไปนี้ลงในไฟล์โปรแกรม โค้ดนี้จะรัน `transpile_remote()` กับทุก `circuits` ที่ส่งเข้ามา และคืน `transpiled_circuits` เป็นผลลัพธ์:

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

<span id="upload-to-qiskit-serverless"></span>
### Authenticate กับ Qiskit Serverless
ใช้ `qiskit-ibm-catalog` เพื่อ authenticate กับ `QiskitServerless` ด้วย API key (สามารถใช้ API key ของ `QiskitRuntimeService` หรือสร้าง API key ใหม่บน [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com))

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

### Verify upload

To check if it successfully uploaded, use `serverless.list()`, as in the following code:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

## Next steps

<Admonition type="info" title="Recommendations">

- Learn how to pass inputs and run your program remotely in the [Run your first Qiskit Serverless workload remotely](/docs/guides/serverless-run-first-workload) topic.

</Admonition>